# 🇰🇷 Hello Clova Agent — HyperCLOVA X SEED Think-14B

**한국어 프롬프트 한 줄 → Reveal.js 발표 슬라이드 자동 생성**  
**국산 LLM 전용 버전 | LLM: HyperCLOVA X SEED Think-14B (Naver)**

---

## 이 노트북의 목적

Gamma, SkyAI 같은 발표 자동 생성 서비스의 **핵심 파이프라인을 직접 구현하고 체험**합니다.  
공공기관·기업 환경에서 **국산 LLM 사용 요건**을 충족하기 위해 Naver HyperCLOVA X SEED를 사용합니다.

| 개념 | 이 프로젝트에서 | 코드 위치 |
|------|----------------|----------|
| **API 서버** | vLLM이 LLM을 HTTP API로 제공 | 셀 1~2 |
| **앱 서버** | LangGraph 4-노드 에이전트 파이프라인 | `agent/graph.py` |
| **웹 서버** | Gradio가 브라우저 요청을 처리 | `ui/app.py` |

---

## 사전 준비

> **Colab 런타임 설정 필수**: 런타임 → 런타임 유형 변경 → **T4 GPU** (VRAM 15GB) 선택  
> A100 (40GB) 환경이라면 더 빠르게 동작합니다.

| 항목 | 내용 |
|------|------|
| 모델 | `naver-hyperclovax/HyperCLOVA-X-SEED-Think-14B` |
| 양자화 | 4-bit bitsandbytes (bf16 원본 ~28GB → ~8-10GB로 압축) |
| 서빙 | vLLM (OpenAI-compatible API) |
| 소요 시간 | 최초 실행 시 모델 다운로드 약 10~20분 |

---

## 실행 방법

**셀을 위에서 아래로 순서대로 실행하세요.** (단축키: `Shift + Enter`)  
셀 1에서 서버를 백그라운드로 시작하고, 셀 2~3에서 코드를 준비하는 동안 모델이 로딩됩니다.

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# 셀 1/5  ▌ 환경 진단 + CUDA 13 런타임 설치 + 의존성 설치 + vLLM 서버 시작
#
# [HyperCLOVAX-SEED-Think-14B 공식 요구사항]
#   pip install transformers>=5.9.0  (먼저)
#   pip install vllm
#   모델 ID: naver-hyperclovax/HyperCLOVAX-SEED-Think-14B
#
# [이 셀의 흐름]
#   CUDA 13 런타임 설치 → pip 설치 → vLLM import 검증 → 서버 시작
#   각 단계 실패 시 즉시 중단 (5분 기다린 후 실패 사이클 방지)
#
# [이전 이슈 및 해결책]
#   증상: vllm/_C.abi3.so → libcudart.so.13 ELF VERNEED 불일치
#   원인: vLLM PyPI Python 3.12 휠은 CUDA 13 컴파일, Colab은 CUDA 12.8 제공
#   시도한 방법: libcudart.so.12 → .so.13 심볼릭 링크 — 파일명은 해결되지만
#               ELF 버전 태그(@@libcudart.so.13)는 해결 불가
#   최종 해결: apt-get install cuda-cudart-13-0 으로 실제 CUDA 13 런타임 설치
#              T4 드라이버(580.82+)가 CUDA 13을 지원하므로 가능
# ══════════════════════════════════════════════════════════════════════

import subprocess, os, sys, torch

HCX_MODEL = "naver-hyperclovax/HyperCLOVAX-SEED-Think-14B"

# ── 1) GPU / CUDA 환경 진단 ───────────────────────────────────────────
print("🔍 CUDA 환경 진단 중...")
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader \
    2>/dev/null || echo "  ⚠️ GPU 없음 — 런타임 → T4 GPU 선택 후 재실행하세요"

cuda_ver = torch.version.cuda or "unknown"
print(f"  PyTorch : {torch.__version__}")
print(f"  CUDA    : {cuda_ver}")

# ── 2) CUDA 13 런타임 설치 ───────────────────────────────────────────
# vLLM _C.abi3.so 는 @@libcudart.so.13 ELF 버전 태그를 요구합니다.
# 심볼릭 링크로는 파일명만 해결되고 버전 태그는 해결 불가 → 실제 패키지 설치 필요.
# T4 드라이버가 CUDA 13을 지원하므로 apt 소스에서 바로 받을 수 있습니다.
print("\n🔧 CUDA 13 런타임 설치 중... (약 10초)")
result = subprocess.run(
    ["apt-get", "install", "-y", "-q", "cuda-cudart-13-0"],
    capture_output=True, text=True,
)
if result.returncode != 0:
    print("❌ CUDA 13 런타임 설치 실패:")
    print(result.stderr[-500:])
    raise RuntimeError("cuda-cudart-13-0 설치 실패. 위 로그를 확인하세요.")
print("  ✅ cuda-cudart-13-0 설치 완료")

# /usr/local/cuda → /usr/local/cuda-13.0 (update-alternatives 자동 처리됨)
# LD_LIBRARY_PATH에 명시적으로 추가해 subprocess 환경에서도 인식되도록 함
cuda13_lib = "/usr/local/cuda-13.0/lib64"
ld = os.environ.get("LD_LIBRARY_PATH", "")
if cuda13_lib not in ld:
    os.environ["LD_LIBRARY_PATH"] = f"{cuda13_lib}:{ld}"
subprocess.run(["ldconfig"], capture_output=True)
print(f"  🔗 LD_LIBRARY_PATH += {cuda13_lib}")

# ── 3) 의존성 설치 (순서 중요: transformers 먼저) ────────────────────
print("\n📦 [1/2] transformers>=5.9.0 설치 중...")
subprocess.check_call([sys.executable, "-m", "pip", "install",
                       "transformers>=5.9.0", "-q"])

print("📦 [2/2] vLLM + bitsandbytes 설치 중...")
subprocess.check_call([sys.executable, "-m", "pip", "install",
                       "vllm", "bitsandbytes", "-q"])
print("✅ 의존성 설치 완료")

# ── 4) vLLM import 사전 검증 ─────────────────────────────────────────
print("\n🔍 vLLM import 사전 검증 중... (최대 30초)")
env = os.environ.copy()
check = subprocess.run(
    [sys.executable, "-c",
     "from vllm.entrypoints.openai import api_server; print('OK')"],
    capture_output=True, text=True, timeout=60, env=env,
)
if check.returncode != 0:
    print("❌ vLLM import 실패 — 서버 시작을 중단합니다.")
    lines = (check.stderr or check.stdout).strip().splitlines()
    for line in lines[-15:]:
        print(f"   {line}")
    raise RuntimeError("vLLM import 실패. 위 로그를 확인하세요.")
print("  ✅ vLLM import 사전 검증 통과")

# ── 5) vLLM 서버 백그라운드 시작 ────────────────────────────────────
print(f"\n🚀 vLLM API 서버 시작: {HCX_MODEL}")
print("   4-bit 양자화 (bitsandbytes) — VRAM ~8-10GB 사용")
print("   모델 로딩 약 10~20분 소요 (셀 2~3 실행하면서 대기)")

vllm_proc = subprocess.Popen(
    [
        sys.executable, "-m", "vllm.entrypoints.openai.api_server",
        "--model",                   HCX_MODEL,
        "--host",                    "0.0.0.0",
        "--port",                    "8000",
        "--dtype",                   "bfloat16",
        "--quantization",            "bitsandbytes",
        "--load-format",             "bitsandbytes",
        "--max-model-len",           "4096",
        "--gpu-memory-utilization",  "0.90",
        "--trust-remote-code",
        "--served-model-name",       HCX_MODEL,
    ],
    env=env,
    stdout=open("/tmp/vllm.log", "w"),
    stderr=subprocess.STDOUT,
)

os.environ["VLLM_PID"] = str(vllm_proc.pid)
print(f"\n✅ vLLM 프로세스 시작 (PID: {vllm_proc.pid})")
print("   → 셀 2, 3 실행 후 셀 4에서 준비 완료 대기")
print("   → 지금 바로 로그 확인: !tail -f /tmp/vllm.log")

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# 셀 2/5  ▌ 프로젝트 코드 준비 (git clone / git pull)
#
# 처음 실행: 저장소 전체를 내려받습니다 (git clone)
# 재실행 시: 최신 코드로 업데이트합니다  (git pull)
# ══════════════════════════════════════════════════════════════════════

import os

REPO_URL = "https://github.com/machine-artisan/Hello-Clova-Agent.git"
REPO_DIR = "Hello-Clova-Agent"

if os.path.exists(REPO_DIR):
    print("🔄 최신 코드로 업데이트 중...")
    !git -C {REPO_DIR} pull
else:
    print("📂 프로젝트 다운로드 중...")
    !git clone {REPO_URL}

os.chdir(REPO_DIR)
print(f"\n✅ 현재 경로: {os.getcwd()}")

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# 셀 3/5  ▌ 의존성 설치 + 환경변수 설정
#
# [pip 충돌 경고 안내]
#   vLLM이 starlette/opentelemetry 버전을 변경하면서
#   Colab 사전설치 패키지(google-adk, prometheus-fastapi-instrumentator)와
#   충돌 경고가 발생합니다.
#   → 이 충돌은 Colab 자체 패키지 문제이며, 이 프로젝트와 무관합니다.
# ══════════════════════════════════════════════════════════════════════

import os, sys, subprocess, importlib

# ── 1) 프로젝트 의존성 설치 ──────────────────────────────────────────
print("📦 프로젝트 의존성 설치 중 (requirements.txt)...")
result = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-r", "requirements.txt", "-q"],
    capture_output=True, text=True
)

# 실제 오류 필터링:
# Colab 기존 패키지(google-adk, prometheus 등) vs vLLM 버전 충돌 경고와
# pip resolver preamble 은 우리 프로젝트와 무관하므로 무시합니다.
IGNORE_PATTERNS = [
    "google-adk", "prometheus-fastapi", "opentelemetry", "starlette",
    "dependency resolver",   # pip preamble ("ERROR: pip's dependency resolver...")
    "behaviour is the source",
    "does not currently take",
]
real_errors = [
    line for line in (result.stdout + result.stderr).splitlines()
    if ("error" in line.lower() or "failed" in line.lower())
    and not any(pat in line.lower() for pat in IGNORE_PATTERNS)
    and line.strip()
]

if real_errors:
    print("❌ 프로젝트 패키지 설치 실패:")
    for e in real_errors:
        print(f"   {e}")
else:
    print("   ✅ requirements.txt 설치 완료")
    print("   (Colab 자체 패키지 충돌 경고는 이 프로젝트와 무관 — 무시)")

# ── 2) 핵심 패키지 import 검증 ──────────────────────────────────────
print("\n🔍 핵심 패키지 검증:")
VERIFY = [
    ("langgraph",     "langgraph"),
    ("openai",        "openai"),
    ("gradio",        "gradio"),
    ("python-dotenv", "dotenv"),
]
all_ok = True
for pkg_name, import_name in VERIFY:
    try:
        mod = importlib.import_module(import_name)
        ver = getattr(mod, "__version__", "설치됨")
        print(f"   ✅ {pkg_name} ({ver})")
    except ImportError as e:
        print(f"   ❌ {pkg_name}: {e}")
        all_ok = False

if not all_ok:
    raise RuntimeError("핵심 패키지 import 실패 — 위 오류를 확인하세요")

# ── 3) 환경변수 설정 ─────────────────────────────────────────────────
HCX_MODEL = "naver-hyperclovax/HyperCLOVAX-SEED-Think-14B"
os.environ["LLM_API_BASE"] = "http://localhost:8000/v1"
os.environ["LLM_API_KEY"]  = "EMPTY"
os.environ["LLM_MODEL"]    = HCX_MODEL

print(f"\n✅ 환경변수 설정 완료")
print(f"   LLM_API_BASE = {os.environ['LLM_API_BASE']}")
print(f"   LLM_MODEL    = {os.environ['LLM_MODEL']}")

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# 셀 4/5  ▌ vLLM 서버 준비 완료 대기
#
# 프로세스 종료 감지 → 즉시 전체 로그 출력 (기다리지 않음)
# 60초마다 /tmp/vllm.log 최근 30줄 출력
# ══════════════════════════════════════════════════════════════════════

import time, urllib.request, subprocess, os, signal

HEALTH_URL = "http://localhost:8000/health"
MAX_WAIT   = 1200  # 20분 (모델 로딩 포함)
INTERVAL   = 10
LOG_EVERY  = 60

vllm_pid = int(os.environ.get("VLLM_PID", "0"))

def pid_alive(pid):
    try:
        os.kill(pid, 0)
        return True
    except (ProcessLookupError, PermissionError):
        return False

def show_log(n=30, label=""):
    result = subprocess.run(["tail", f"-{n}", "/tmp/vllm.log"],
                            capture_output=True, text=True)
    if label:
        print(label)
    print(result.stdout or "  (로그 없음)")

print(f"⏳ vLLM 서버 준비 대기 중 (최대 {MAX_WAIT//60}분, PID: {vllm_pid})\n")

elapsed      = 0
next_log_at  = LOG_EVERY
server_ready = False

while elapsed < MAX_WAIT:
    # ── 프로세스가 이미 종료됐으면 즉시 실패 처리 ──────────────────
    if vllm_pid and not pid_alive(vllm_pid):
        print(f"\n❌ vLLM 프로세스(PID {vllm_pid})가 {elapsed}s에 비정상 종료")
        show_log(n=100, label="── /tmp/vllm.log 전체 (최근 100줄) ──")
        break

    # ── health check ────────────────────────────────────────────────
    try:
        urllib.request.urlopen(HEALTH_URL, timeout=3)
        server_ready = True
        break
    except Exception:
        pass

    print(f"  [{elapsed:3d}s] 대기 중...", end="\r")

    if elapsed >= next_log_at:
        show_log(n=30, label=f"\n── [{elapsed}s] /tmp/vllm.log 최근 30줄 ──")
        next_log_at += LOG_EVERY

    time.sleep(INTERVAL)
    elapsed += INTERVAL

if server_ready:
    print(f"\n✅ vLLM 서버 준비 완료! ({elapsed//60}분 {elapsed%60}초)")
    print("   → 셀 5를 실행해 에이전트를 시작하세요")
elif not (vllm_pid and not pid_alive(vllm_pid)):
    print(f"\n❌ {MAX_WAIT//60}분 내 서버 시작 실패. 전체 로그:")
    show_log(n=100)

## 🏗️ 시스템 아키텍처 — 에이전트가 동작하는 방식

```
┌─────────────────────────────────────────────────────────┐
│                   브라우저 (여러분의 PC)                 │
└───────────────────────┬─────────────────────────────────┘
                        │  HTTP (공개 URL)
┌───────────────────────▼─────────────────────────────────┐
│         🌐 웹서버  —  Gradio  (포트 7860)               │
│         ui/app.py  :  브라우저 ↔ 에이전트 연결          │
└───────────────────────┬─────────────────────────────────┘
                        │  Python 함수 호출
┌───────────────────────▼─────────────────────────────────┐
│         ⚙️  앱서버  —  LangGraph 파이프라인             │
│   Node1 input_parser → Node2 outline_generator          │
│   Node3 slide_writer → Node4 html_renderer              │
└───────────────────────┬─────────────────────────────────┘
                        │  HTTP (OpenAI API 형식)
┌───────────────────────▼─────────────────────────────────┐
│         🤖 API서버  —  vLLM  (포트 8000)                │
│         /v1/chat/completions  :  LLM 추론 제공          │
└───────────────────────┬─────────────────────────────────┘
                        │  GPU (4-bit bitsandbytes)
┌───────────────────────▼─────────────────────────────────┐
│   🇰🇷 LLM  —  HyperCLOVA X SEED Think-14B  (Naver)    │
│         VRAM ~8-10GB  |  Colab T4 (15GB) 가능           │
└─────────────────────────────────────────────────────────┘
```

| 계층 | 기술 | 역할 |
|------|------|------|
| **웹서버** | Gradio | 브라우저의 HTTP 요청 수신 + 응답 |
| **앱서버** | LangGraph | 비즈니스 로직 (에이전트 파이프라인 실행) |
| **API서버** | vLLM | LLM 추론을 REST API로 제공 |
| **LLM** | HyperCLOVA X SEED Think-14B | 실제 텍스트 생성 (국산 LLM) |

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# 셀 5/5  ▌ 에이전트 실행 🚀
#
# 셀 실행 후 출력되는 공개 URL을 브라우저에서 열어주세요.
#   예) Running on public URL: https://xxxx.gradio.live
#
# 중지하려면: 셀 왼쪽의 ■ 버튼 클릭
# ══════════════════════════════════════════════════════════════════════

print("🚀 Hello Clova Agent (HyperCLOVA X SEED Think-14B) 시작 중...")
print("   아래에 공개 URL이 출력되면 브라우저에서 접속하세요 👇")
print()

!python ui/app.py

---

## 🔬 Phase 2 선택 실행 — 임베딩 모델 (RAG 준비)

아래 셀은 **Phase 2 RAG 연동**을 위한 임베딩 모델 설정입니다.  
Phase 1 체험만 원하면 실행하지 않아도 됩니다.

| 항목 | 내용 |
|------|------|
| 모델 | `BAAI/bge-m3` (다국어 임베딩, 한국어 우수) |
| 용도 | 외부 문서를 벡터로 변환하여 RAG 검색에 활용 |
| 크기 | ~570 MB |

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# [선택] Phase 2  ▌ 임베딩 모델 준비 (RAG용)
#
# [개념] 임베딩(Embedding)이란?
#   텍스트를 의미를 담은 숫자 벡터로 변환하는 기술입니다.
#   비슷한 의미의 문장은 벡터 공간에서 가까운 위치에 배치되어
#   유사도 검색(RAG의 핵심)이 가능해집니다.
# ══════════════════════════════════════════════════════════════════════

import os

print("📦 임베딩 관련 패키지 설치 중...")
!pip install langchain-huggingface sentence-transformers -q

from langchain_huggingface import HuggingFaceEmbeddings
from sentence_transformers import SentenceTransformer

EMBED_MODEL = "BAAI/bge-m3"

if not os.path.exists(EMBED_MODEL):
    print(f"📥 임베딩 모델 다운로드 중: {EMBED_MODEL}")
    model = SentenceTransformer(EMBED_MODEL)
    model.save(EMBED_MODEL)
    print("✅ 다운로드 완료 — 로컬 캐시에 저장됨")
else:
    print(f"✅ 캐시된 모델 사용: {EMBED_MODEL}")

embeddings = HuggingFaceEmbeddings(
    model_name=EMBED_MODEL,
    model_kwargs={"device": "cuda", "trust_remote_code": True},
    encode_kwargs={"normalize_embeddings": True},
)

test_vec = embeddings.embed_query("안녕하세요, 임베딩 테스트입니다.")
print(f"\n✅ 임베딩 모델 준비 완료")
print(f"   벡터 차원: {len(test_vec)}")
print(f"   샘플 벡터 (앞 5개): {[round(v, 4) for v in test_vec[:5]]}")

---

## 📚 더 알아보기

### 코드 구조 탐색

```
Hello-Clova-Agent/
├── agent/
│   ├── state.py              ← 파이프라인이 공유하는 데이터 구조
│   ├── llm.py                ← vLLM API 클라이언트 (OpenAI 호환)
│   ├── graph.py              ← LangGraph 파이프라인 조립
│   └── nodes/
│       ├── input_parser.py   ← Node 1: 입력 분석
│       ├── outline_generator.py  ← Node 2: 목차 생성 (LLM)
│       ├── slide_writer.py   ← Node 3: 내용 작성 (LLM)
│       └── html_renderer.py  ← Node 4: HTML 변환
└── ui/app.py                 ← Gradio 웹 UI
```

### HyperCLOVA X SEED 모델 라인업

| 모델 | 방식 | VRAM | 특징 |
|------|------|------|------|
| **Think-14B** ← 이 노트북 | 4-bit vLLM | ~8-10 GB | T4 가능, 고품질 |
| Instruct-3B | fp16 vLLM | ~8 GB | 빠른 추론 |
| Think-32B | 4-bit vLLM | ~18 GB | 최고 품질, A100 필요 |

---
*Hello Clova Agent · Phase 1 MVP · LangGraph + vLLM + HyperCLOVA X SEED Think-14B + Reveal.js*